# Retail Sales Analysis using Pandas and SQL

**Title:** Customer Purchase and Sales Performance Analysis

**Business Scenario:** A retail company sells products across cities via Online and Offline channels. As a data analyst, we analyze sales, customer behavior, product performance, and revenue trends using **SQL** (via Python's built-in `sqlite3`, so the notebook is fully self-contained and runnable without an external database server) and **Pandas**.

This notebook is split into:
- **Part A — SQL Hands-on Tasks (1–10)**
- **Part B — Pandas Hands-on Tasks (1–10)**
- **Business Insights Summary**

Each task includes a **Problem Statement**, an **Approach**, and a fully commented code cell.


## Setup: Sample Data

**Problem Statement:** Before any SQL or Pandas analysis, we need the three source datasets — Customers, Products, Orders — available in Python.

**Approach:** Define each table's data as a list of tuples/dicts matching the given sample data. This raw data will be used both to populate the SQLite database (Part A) and to build Pandas DataFrames (Part B).


In [1]:
import sqlite3
import pandas as pd
import numpy as np

# ---- Customers data ----
customers_data = [
    ("C001", "Rahul Sharma", "Patna", 32, "Male"),
    ("C002", "Priya Singh", "Delhi", 28, "Female"),
    ("C003", "Amit Kumar", "Kolkata", 35, "Male"),
    ("C004", "Sneha Verma", "Pune", 30, "Female"),
    ("C005", "Rohit Raj", "Patna", 40, "Male"),
    ("C006", "Neha Gupta", "Delhi", 26, "Female"),
    ("C007", "Ankit Sinha", "Mumbai", 38, "Male"),
    ("C008", "Riya Das", "Kolkata", 24, "Female"),
]

# ---- Products data ----
products_data = [
    ("P001", "Laptop", "Electronics", 55000),
    ("P002", "Mobile Phone", "Electronics", 25000),
    ("P003", "Office Chair", "Furniture", 7000),
    ("P004", "Headphones", "Electronics", 3000),
    ("P005", "Study Table", "Furniture", 12000),
    ("P006", "Shoes", "Fashion", 4000),
    ("P007", "Backpack", "Fashion", 2500),
]

# ---- Orders data ----
orders_data = [
    ("O001", "C001", "P002", "2026-01-10", 2, "Online"),
    ("O002", "C002", "P001", "2026-01-15", 1, "Offline"),
    ("O003", "C003", "P004", "2026-02-05", 3, "Online"),
    ("O004", "C004", "P003", "2026-02-12", 2, "Offline"),
    ("O005", "C005", "P006", "2026-03-01", 4, "Online"),
    ("O006", "C001", "P004", "2026-03-08", 2, "Online"),
    ("O007", "C006", "P005", "2026-03-18", 1, "Offline"),
    ("O008", "C007", "P001", "2026-04-02", 1, "Online"),
    ("O009", "C008", "P007", "2026-04-10", 5, "Online"),
    ("O010", "C003", "P002", "2026-04-22", 1, "Offline"),
    ("O011", "C005", "P003", "2026-05-05", 1, "Offline"),
    ("O012", "C002", "P004", "2026-05-15", 4, "Online"),
]

print("Sample data defined:", len(customers_data), "customers,",
      len(products_data), "products,", len(orders_data), "orders")

Sample data defined: 8 customers, 7 products, 12 orders


# Part A: SQL Hands-on Tasks

**Approach note:** We use Python's built-in `sqlite3` module to create an in-memory SQL database. This lets us write and execute *real SQL* (CREATE TABLE, INSERT, SELECT with JOIN/GROUP BY) directly inside the notebook without needing an external database server — the SQL syntax itself is standard and portable to MySQL/PostgreSQL.


## Task 1: Create Tables

**Problem Statement:** Create three tables — `customers`, `products`, `orders` — with proper columns and suitable data types.

**Approach:** Use `CREATE TABLE` statements. Primary keys are the ID columns; `orders` includes foreign keys referencing `customers` and `products` to enforce relational integrity.


In [2]:
# Connect to an in-memory SQLite database (self-contained, no server needed)
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Create customers table
cursor.execute("""
CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY,
    customer_name TEXT NOT NULL,
    city TEXT,
    age INTEGER,
    gender TEXT
)
""")

# Create products table
cursor.execute("""
CREATE TABLE products (
    product_id TEXT PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT,
    price REAL
)
""")

# Create orders table with foreign keys to customers and products
cursor.execute("""
CREATE TABLE orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT,
    product_id TEXT,
    order_date TEXT,
    quantity INTEGER,
    sales_channel TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
)
""")

conn.commit()
print("Tables created: customers, products, orders")

Tables created: customers, products, orders


## Task 2: Insert Data

**Problem Statement:** Insert the given sample data into all three tables.

**Approach:** Use parameterized `INSERT INTO ... VALUES (?, ?, ...)` statements with `executemany()` for bulk insertion — this is safer than string formatting and avoids SQL injection issues.


In [3]:
cursor.executemany(
    "INSERT INTO customers VALUES (?, ?, ?, ?, ?)", customers_data
)
cursor.executemany(
    "INSERT INTO products VALUES (?, ?, ?, ?)", products_data
)
cursor.executemany(
    "INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?)", orders_data
)
conn.commit()

print("Rows inserted -> customers:", cursor.execute("SELECT COUNT(*) FROM customers").fetchone()[0])
print("Rows inserted -> products:", cursor.execute("SELECT COUNT(*) FROM products").fetchone()[0])
print("Rows inserted -> orders:", cursor.execute("SELECT COUNT(*) FROM orders").fetchone()[0])

Rows inserted -> customers: 8
Rows inserted -> products: 7
Rows inserted -> orders: 12


## Task 3: View All Orders with Customer and Product Details

**Problem Statement:** Display order_id, customer_name, city, product_name, category, quantity, price, total_amount, sales_channel, order_date. `total_amount = quantity × price`.

**Approach:** JOIN `orders` with `customers` (on customer_id) and `products` (on product_id), then compute `total_amount` as a calculated column in the SELECT clause.


In [4]:
query_task3 = """
SELECT
    o.order_id,
    c.customer_name,
    c.city,
    p.product_name,
    p.category,
    o.quantity,
    p.price,
    (o.quantity * p.price) AS total_amount,
    o.sales_channel,
    o.order_date
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
ORDER BY o.order_id
"""

df_task3 = pd.read_sql_query(query_task3, conn)
display(df_task3)

,order_id,customer_name,city,product_name,category,quantity,price,total_amount,sales_channel,order_date
0,O001,Rahul Sharma,Patna,Mobile Phone,Electronics,2,25000.0,50000.0,Online,2026-01-10
1,O002,Priya Singh,Delhi,Laptop,Electronics,1,55000.0,55000.0,Offline,2026-01-15
2,O003,Amit Kumar,Kolkata,Headphones,Electronics,3,3000.0,9000.0,Online,2026-02-05
3,O004,Sneha Verma,Pune,Office Chair,Furniture,2,7000.0,14000.0,Offline,2026-02-12
4,O005,Rohit Raj,Patna,Shoes,Fashion,4,4000.0,16000.0,Online,2026-03-01
5,O006,Rahul Sharma,Patna,Headphones,Electronics,2,3000.0,6000.0,Online,2026-03-08
6,O007,Neha Gupta,Delhi,Study Table,Furniture,1,12000.0,12000.0,Offline,2026-03-18
7,O008,Ankit Sinha,Mumbai,Laptop,Electronics,1,55000.0,55000.0,Online,2026-04-02
8,O009,Riya Das,Kolkata,Backpack,Fashion,5,2500.0,12500.0,Online,2026-04-10
9,O010,Amit Kumar,Kolkata,Mobile Phone,Electronics,1,25000.0,25000.0,Offline,2026-04-22


## Task 4: Find Total Revenue

**Problem Statement:** Calculate the total revenue generated by the company (`total_revenue`).

**Approach:** Join orders with products, compute quantity × price per row, and use `SUM()` to aggregate across all rows.


In [5]:
query_task4 = """
SELECT SUM(o.quantity * p.price) AS total_revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
"""

df_task4 = pd.read_sql_query(query_task4, conn)
display(df_task4)

,total_revenue
0,273500.0


## Task 5: Revenue by City

**Problem Statement:** Find total revenue generated from each city (`city`, `total_revenue`).

**Approach:** Join orders → customers (for city) → products (for price), then `GROUP BY city` and `SUM` the revenue per group.


In [6]:
query_task5 = """
SELECT
    c.city,
    SUM(o.quantity * p.price) AS total_revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
GROUP BY c.city
ORDER BY total_revenue DESC
"""

df_task5 = pd.read_sql_query(query_task5, conn)
display(df_task5)

,city,total_revenue
0,Patna,79000.0
1,Delhi,79000.0
2,Mumbai,55000.0
3,Kolkata,46500.0
4,Pune,14000.0


## Task 6: Best-Selling Product

**Problem Statement:** Find the product with the highest quantity sold (`product_name`, `total_quantity_sold`).

**Approach:** Join orders with products, `GROUP BY product_name`, `SUM(quantity)`, sort descending, and take the top row with `ORDER BY ... LIMIT 1`.


In [7]:
query_task6 = """
SELECT
    p.product_name,
    SUM(o.quantity) AS total_quantity_sold
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_quantity_sold DESC
LIMIT 1
"""

df_task6 = pd.read_sql_query(query_task6, conn)
display(df_task6)

# Bonus: full ranking of all products by quantity sold
query_task6_all = """
SELECT p.product_name, SUM(o.quantity) AS total_quantity_sold
FROM orders o JOIN products p ON o.product_id = p.product_id
GROUP BY p.product_name ORDER BY total_quantity_sold DESC
"""
display(pd.read_sql_query(query_task6_all, conn))

,product_name,total_quantity_sold
0,Headphones,9


,product_name,total_quantity_sold
0,Headphones,9
1,Backpack,5
2,Shoes,4
3,Office Chair,3
4,Mobile Phone,3
5,Laptop,2
6,Study Table,1


## Task 7: Category-wise Revenue

**Problem Statement:** Find total revenue for each product category (`category`, `total_revenue`).

**Approach:** Join orders with products, `GROUP BY category`, `SUM(quantity * price)`.


In [8]:
query_task7 = """
SELECT
    p.category,
    SUM(o.quantity * p.price) AS total_revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC
"""

df_task7 = pd.read_sql_query(query_task7, conn)
display(df_task7)

,category,total_revenue
0,Electronics,212000.0
1,Furniture,33000.0
2,Fashion,28500.0


## Task 8: Online vs Offline Sales

**Problem Statement:** Compare total revenue from Online and Offline sales channels (`sales_channel`, `total_revenue`).

**Approach:** Join orders with products, `GROUP BY sales_channel`, `SUM(quantity * price)`.


In [9]:
query_task8 = """
SELECT
    o.sales_channel,
    SUM(o.quantity * p.price) AS total_revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY o.sales_channel
ORDER BY total_revenue DESC
"""

df_task8 = pd.read_sql_query(query_task8, conn)
display(df_task8)

,sales_channel,total_revenue
0,Online,160500.0
1,Offline,113000.0


## Task 9: Monthly Revenue Trend

**Problem Statement:** Find monthly revenue from January to May 2026 (`month`, `total_revenue`).

**Approach:** Use SQLite's `strftime('%Y-%m', order_date)` to extract the year-month from the `order_date` text field, then `GROUP BY` that month string and `SUM` revenue.


In [10]:
query_task9 = """
SELECT
    strftime('%Y-%m', o.order_date) AS month,
    SUM(o.quantity * p.price) AS total_revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY month
ORDER BY month
"""

df_task9 = pd.read_sql_query(query_task9, conn)
display(df_task9)

,month,total_revenue
0,2026-01,105000.0
1,2026-02,23000.0
2,2026-03,34000.0
3,2026-04,92500.0
4,2026-05,19000.0


## Task 10: High-Value Customers

**Problem Statement:** Find customers whose total purchase value exceeds ₹50,000 (`customer_name`, `city`, `total_purchase_value`).

**Approach:** Join orders → customers → products, `GROUP BY customer_name, city`, `SUM(quantity * price)`, then filter aggregated groups using `HAVING` (since we filter on an aggregate, not a raw column, `WHERE` cannot be used here).


In [11]:
query_task10 = """
SELECT
    c.customer_name,
    c.city,
    SUM(o.quantity * p.price) AS total_purchase_value
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
GROUP BY c.customer_name, c.city
HAVING SUM(o.quantity * p.price) > 50000
ORDER BY total_purchase_value DESC
"""

df_task10 = pd.read_sql_query(query_task10, conn)
display(df_task10)

,customer_name,city,total_purchase_value
0,Priya Singh,Delhi,67000.0
1,Rahul Sharma,Patna,56000.0
2,Ankit Sinha,Mumbai,55000.0


# Part B: Pandas Hands-on Tasks

**Approach note:** We now reproduce the exact same analysis purely with Pandas — no SQL — to compare/practice both approaches on identical data.


## Task 1: Create DataFrames

**Problem Statement:** Create three Pandas DataFrames: `customers_df`, `products_df`, `orders_df`.

**Approach:** Convert the same raw Python lists used for SQL into DataFrames with explicit column names, ensuring consistency between SQL and Pandas results.


In [12]:
customers_df = pd.DataFrame(customers_data,
    columns=["customer_id", "customer_name", "city", "age", "gender"])

products_df = pd.DataFrame(products_data,
    columns=["product_id", "product_name", "category", "price"])

orders_df = pd.DataFrame(orders_data,
    columns=["order_id", "customer_id", "product_id", "order_date", "quantity", "sales_channel"])

display(customers_df)
display(products_df)
display(orders_df)

,customer_id,customer_name,city,age,gender
0,C001,Rahul Sharma,Patna,32,Male
1,C002,Priya Singh,Delhi,28,Female
2,C003,Amit Kumar,Kolkata,35,Male
3,C004,Sneha Verma,Pune,30,Female
4,C005,Rohit Raj,Patna,40,Male
5,C006,Neha Gupta,Delhi,26,Female
6,C007,Ankit Sinha,Mumbai,38,Male
7,C008,Riya Das,Kolkata,24,Female


,product_id,product_name,category,price
0,P001,Laptop,Electronics,55000
1,P002,Mobile Phone,Electronics,25000
2,P003,Office Chair,Furniture,7000
3,P004,Headphones,Electronics,3000
4,P005,Study Table,Furniture,12000
5,P006,Shoes,Fashion,4000
6,P007,Backpack,Fashion,2500


,order_id,customer_id,product_id,order_date,quantity,sales_channel
0,O001,C001,P002,2026-01-10,2,Online
1,O002,C002,P001,2026-01-15,1,Offline
2,O003,C003,P004,2026-02-05,3,Online
3,O004,C004,P003,2026-02-12,2,Offline
4,O005,C005,P006,2026-03-01,4,Online
5,O006,C001,P004,2026-03-08,2,Online
6,O007,C006,P005,2026-03-18,1,Offline
7,O008,C007,P001,2026-04-02,1,Online
8,O009,C008,P007,2026-04-10,5,Online
9,O010,C003,P002,2026-04-22,1,Offline


## Task 2: Merge DataFrames

**Problem Statement:** Merge all three DataFrames into one final sales DataFrame with customer, product, and order details.

**Approach:** Use `pd.merge()` twice — first `orders_df` with `customers_df` on `customer_id`, then the result with `products_df` on `product_id` — mirroring the SQL JOINs from Part A.


In [13]:
sales_df = orders_df.merge(customers_df, on="customer_id", how="left")
sales_df = sales_df.merge(products_df, on="product_id", how="left")

display(sales_df)

,order_id,customer_id,product_id,order_date,quantity,sales_channel,customer_name,city,age,gender,product_name,category,price
0,O001,C001,P002,2026-01-10,2,Online,Rahul Sharma,Patna,32,Male,Mobile Phone,Electronics,25000
1,O002,C002,P001,2026-01-15,1,Offline,Priya Singh,Delhi,28,Female,Laptop,Electronics,55000
2,O003,C003,P004,2026-02-05,3,Online,Amit Kumar,Kolkata,35,Male,Headphones,Electronics,3000
3,O004,C004,P003,2026-02-12,2,Offline,Sneha Verma,Pune,30,Female,Office Chair,Furniture,7000
4,O005,C005,P006,2026-03-01,4,Online,Rohit Raj,Patna,40,Male,Shoes,Fashion,4000
5,O006,C001,P004,2026-03-08,2,Online,Rahul Sharma,Patna,32,Male,Headphones,Electronics,3000
6,O007,C006,P005,2026-03-18,1,Offline,Neha Gupta,Delhi,26,Female,Study Table,Furniture,12000
7,O008,C007,P001,2026-04-02,1,Online,Ankit Sinha,Mumbai,38,Male,Laptop,Electronics,55000
8,O009,C008,P007,2026-04-10,5,Online,Riya Das,Kolkata,24,Female,Backpack,Fashion,2500
9,O010,C003,P002,2026-04-22,1,Offline,Amit Kumar,Kolkata,35,Male,Mobile Phone,Electronics,25000


## Task 3: Create Total Amount Column

**Problem Statement:** Add a new column `total_amount = quantity × price`.

**Approach:** Directly multiply the two existing columns element-wise (vectorized operation) and assign the result as a new column.


In [14]:
sales_df["total_amount"] = sales_df["quantity"] * sales_df["price"]

display(sales_df[["order_id", "customer_name", "city", "product_name", "category",
                   "quantity", "price", "total_amount", "sales_channel", "order_date"]])

,order_id,customer_name,city,product_name,category,quantity,price,total_amount,sales_channel,order_date
0,O001,Rahul Sharma,Patna,Mobile Phone,Electronics,2,25000,50000,Online,2026-01-10
1,O002,Priya Singh,Delhi,Laptop,Electronics,1,55000,55000,Offline,2026-01-15
2,O003,Amit Kumar,Kolkata,Headphones,Electronics,3,3000,9000,Online,2026-02-05
3,O004,Sneha Verma,Pune,Office Chair,Furniture,2,7000,14000,Offline,2026-02-12
4,O005,Rohit Raj,Patna,Shoes,Fashion,4,4000,16000,Online,2026-03-01
5,O006,Rahul Sharma,Patna,Headphones,Electronics,2,3000,6000,Online,2026-03-08
6,O007,Neha Gupta,Delhi,Study Table,Furniture,1,12000,12000,Offline,2026-03-18
7,O008,Ankit Sinha,Mumbai,Laptop,Electronics,1,55000,55000,Online,2026-04-02
8,O009,Riya Das,Kolkata,Backpack,Fashion,5,2500,12500,Online,2026-04-10
9,O010,Amit Kumar,Kolkata,Mobile Phone,Electronics,1,25000,25000,Offline,2026-04-22


## Task 4: Calculate Total Revenue

**Problem Statement:** Find the total revenue using Pandas.

**Approach:** Sum the `total_amount` column with `.sum()`.


In [15]:
total_revenue = sales_df["total_amount"].sum()
print(f"Total Revenue: {total_revenue}")

Total Revenue: 273500


## Task 5: Revenue by City

**Problem Statement:** Group the data by city and calculate total revenue.

**Approach:** Use `.groupby("city")["total_amount"].sum()`, then sort descending to highlight top cities.


In [16]:
revenue_by_city = sales_df.groupby("city")["total_amount"].sum().sort_values(ascending=False)
print(revenue_by_city)

city
Delhi      79000
Patna      79000
Mumbai     55000
Kolkata    46500
Pune       14000
Name: total_amount, dtype: int64


## Task 6: Product-wise Quantity Sold

**Problem Statement:** Find total quantity sold for each product.

**Approach:** Group by `product_name` and sum the `quantity` column.


In [17]:
quantity_by_product = sales_df.groupby("product_name")["quantity"].sum().sort_values(ascending=False)
print(quantity_by_product)

best_selling_product = quantity_by_product.idxmax()
print(f"\nBest-selling product: {best_selling_product} ({quantity_by_product.max()} units)")

product_name
Headphones      9
Backpack        5
Shoes           4
Office Chair    3
Mobile Phone    3
Laptop          2
Study Table     1
Name: quantity, dtype: int64

Best-selling product: Headphones (9 units)


## Task 7: Category-wise Revenue

**Problem Statement:** Find revenue generated by each category.

**Approach:** Group by `category` and sum `total_amount`.


In [18]:
revenue_by_category = sales_df.groupby("category")["total_amount"].sum().sort_values(ascending=False)
print(revenue_by_category)

category
Electronics    212000
Furniture       33000
Fashion         28500
Name: total_amount, dtype: int64


## Task 8: Sales Channel Analysis

**Problem Statement:** Compare online and offline revenue.

**Approach:** Group by `sales_channel` and sum `total_amount`.


In [19]:
revenue_by_channel = sales_df.groupby("sales_channel")["total_amount"].sum().sort_values(ascending=False)
print(revenue_by_channel)

sales_channel
Online     160500
Offline    113000
Name: total_amount, dtype: int64


## Task 9: Monthly Revenue Trend

**Problem Statement:** Convert `order_date` into date format, extract the month, then calculate monthly revenue.

**Approach:** Use `pd.to_datetime()` to parse the date column, then `.dt.to_period("M")` to extract a year-month period, and group by that period summing `total_amount`.


In [20]:
sales_df["order_date"] = pd.to_datetime(sales_df["order_date"])
sales_df["month"] = sales_df["order_date"].dt.to_period("M")

monthly_revenue = sales_df.groupby("month")["total_amount"].sum()
print(monthly_revenue)

top_month = monthly_revenue.idxmax()
print(f"\nHighest revenue month: {top_month} (₹{monthly_revenue.max()})")

month
2026-01    105000
2026-02     23000
2026-03     34000
2026-04     92500
2026-05     19000
Freq: M, Name: total_amount, dtype: int64

Highest revenue month: 2026-01 (₹105000)


## Task 10: High-Value Customers

**Problem Statement:** Find customers whose total purchase value is more than ₹50,000.

**Approach:** Group by `customer_name` and `city`, sum `total_amount`, then filter the resulting Series/DataFrame using boolean indexing (`> 50000`) — equivalent to SQL's `HAVING`.


In [21]:
customer_totals = sales_df.groupby(["customer_name", "city"])["total_amount"].sum().reset_index()
customer_totals = customer_totals.rename(columns={"total_amount": "total_purchase_value"})

high_value_customers = customer_totals[customer_totals["total_purchase_value"] > 50000] \
    .sort_values(by="total_purchase_value", ascending=False)

display(high_value_customers)

,customer_name,city,total_purchase_value
3,Priya Singh,Delhi,67000
4,Rahul Sharma,Patna,56000
1,Ankit Sinha,Mumbai,55000


# Business Insights Summary

**Problem Statement:** Summarize key business insights derived from both the SQL and Pandas analysis.

**Approach:** Pull the top result from each relevant task's output (city, product, category, channel, month, high-value customers) into one consolidated view.


In [22]:
top_city = revenue_by_city.idxmax()
top_category = revenue_by_category.idxmax()
top_channel = revenue_by_channel.idxmax()

insights = pd.DataFrame({
    "Question": [
        "Which city gives the highest revenue?",
        "Which product sells the most?",
        "Which category performs best?",
        "Is online sales better than offline sales?",
        "Which month generated the highest revenue?",
        "How many high-value customers (>₹50,000)?"
    ],
    "Answer": [
        f"{top_city} (₹{revenue_by_city.max():,.0f})",
        f"{best_selling_product} ({quantity_by_product.max()} units)",
        f"{top_category} (₹{revenue_by_category.max():,.0f})",
        f"{top_channel} leads with ₹{revenue_by_channel.max():,.0f}",
        f"{top_month} (₹{monthly_revenue.max():,.0f})",
        f"{len(high_value_customers)} customer(s): {', '.join(high_value_customers['customer_name'].tolist())}"
    ]
})

display(insights)

,Question,Answer
0,Which city gives the highest revenue?,"Delhi (₹79,000)"
1,Which product sells the most?,Headphones (9 units)
2,Which category performs best?,"Electronics (₹212,000)"
3,Is online sales better than offline sales?,"Online leads with ₹160,500"
4,Which month generated the highest revenue?,"2026-01 (₹105,000)"
5,"How many high-value customers (>₹50,000)?","3 customer(s): Priya Singh, Rahul Sharma, Anki..."


## Final Deliverables Recap

This notebook covers all required deliverables:
- SQL table creation queries (Task A1)
- SQL insert queries (Task A2)
- SQL analysis queries (Tasks A3–A10)
- Pandas DataFrame creation (Task B1) and merge (Task B2)
- Pandas analysis output (Tasks B3–B10)
- Business insights summary (final section)

Close the database connection to clean up resources.


In [23]:
conn.close()
print("Database connection closed. Analysis complete.")

Database connection closed. Analysis complete.
